# 1.5 Verlustfunktionen & Gradientenabstieg

Begleit-Notebook zu **Lektion 1.5**: Verlustfunktionen, Gradientenabstieg in 1D und 2D, Lernraten-Experimente und SGD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
rng = np.random.default_rng(42)

## 1. Verlustfunktionen

In [ ]:
def mse(y, y_hat):
    # TODO: mittlerer quadratischer Fehler
    ...

def mae(y, y_hat):
    # TODO: mittlerer absoluter Fehler
    ...

def kreuzentropie(y, p, eps=1e-12):
    # TODO: binäre Kreuzentropie; p mit np.clip in [eps, 1-eps] halten, damit log(0) nicht vorkommt
    ...

y = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print(mse(y, y + np.array([1, -1, 2, -2, 10])), mae(y, y + np.array([1, -1, 2, -2, 10])))  # 22.0  3.2
print(kreuzentropie(np.array([1, 0, 1]), np.array([0.9, 0.1, 0.8])))   # ~0.145
print(kreuzentropie(np.array([1]), np.array([0.01])))                    # sicher und falsch: ~4.6

**Aufgabe 1.2:** Welcher konstante Wert $c$ minimiert MSE bzw. MAE für die Daten unten? Probiere viele Kandidaten aus und vergleiche mit Mittelwert und Median.

In [ ]:
daten = np.array([1, 2, 2, 3, 4, 5, 30], dtype=float)
cs = np.linspace(0, 30, 3001)
# TODO: MSE und MAE für jeden Kandidaten c (Vorhersage = c für alle Punkte)
mse_c = ...
mae_c = ...
print("bestes c für MSE:", cs[mse_c.argmin()], " Mittelwert:", daten.mean())
print("bestes c für MAE:", cs[mae_c.argmin()], " Median:   ", np.median(daten))

## 2. Gradientenabstieg in 1D

**Aufgabe 2.1:** Implementiere den Gradientenabstieg. Die Funktion soll den ganzen Pfad zurückgeben.

In [ ]:
def gradientenabstieg_1d(df, x0, lr, schritte):
    pfad = [x0]
    x = x0
    for _ in range(schritte):
        # TODO: ein Schritt gegen die Ableitung
        ...
        pfad.append(x)
    return np.array(pfad)

f = lambda x: 0.5 * x**2
df = lambda x: x
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
t = np.linspace(-5, 5, 200)
for ax, lr in zip(axes, [0.05, 0.3, 1.8, 2.1]):
    pfad = gradientenabstieg_1d(df, -4.0, lr, 12)
    ax.plot(t, f(t), color="gray")
    ax.plot(pfad, f(pfad), "o-", ms=4)
    ax.set_ylim(0, 13); ax.set_xlim(-5, 5); ax.set_title(f"η = {lr}")
plt.show()

**Aufgabe 2.2:** Für $f(x) = \frac{k}{2}x^2$ divergiert der Gradientenabstieg ab $\eta = 2/k$. Überprüfe das experimentell für $k = 4$:
Wie groß ist $|x|$ nach 50 Schritten für $\eta = 0{,}49$ und $\eta = 0{,}51$?

In [ ]:
k = 4
# TODO: zwei Läufe mit df = k*x, Start 1.0, je 50 Schritte; letzte |x| ausgeben
...

## 3. Gradientenabstieg in 2D und das Zickzack-Problem

**Aufgabe 3.1:** Implementiere die 2D-Version (mit Vektoren!) und vergleiche ein rundes und ein langgezogenes Tal.

In [ ]:
def gradientenabstieg(grad, start, lr, schritte):
    theta = np.array(start, dtype=float)
    pfad = [theta.copy()]
    for _ in range(schritte):
        # TODO: Vektor-Update
        ...
        pfad.append(theta.copy())
    return np.array(pfad)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
xs, ys = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-2.5, 2.5, 200))
for ax, k in zip(axes, [1, 8]):
    grad = lambda th, k=k: np.array([th[0], k * th[1]])
    pfad = gradientenabstieg(grad, [-3.5, 2.0], lr=0.22, schritte=40)
    ax.contour(xs, ys, 0.5 * (xs**2 + k * ys**2), levels=20, alpha=0.6)
    ax.plot(pfad[:, 0], pfad[:, 1], "o-", ms=3, color="tab:orange")
    ax.set_aspect("equal"); ax.set_title(f"Streckung k = {k}: Endpunkt {np.round(pfad[-1], 3)}")
plt.show()

## 4. Lineare Regression mit Gradientenabstieg

Wir erzeugen Daten $y = 3x - 2 + \text{Rauschen}$ und lernen $w$ und $b$.

**Aufgabe 4.1:** Ergänze den Trainingsschritt und zeichne die Lernkurve.

In [ ]:
n = 200
x = rng.uniform(0, 5, n)
y = 3 * x - 2 + rng.normal(0, 1, n)

def train_gd(x, y, lr, epochen):
    w, b = 0.0, 0.0
    verluste = []
    for _ in range(epochen):
        y_hat = w * x + b
        # TODO: Gradienten von MSE nach w und b, dann Update
        ...
        verluste.append(np.mean((w * x + b - y) ** 2))
    return w, b, verluste

w, b, verluste = train_gd(x, y, lr=0.05, epochen=300)
print(f"gelernt: w = {w:.3f}, b = {b:.3f}  (wahr: 3, -2)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x, y, s=8); t = np.linspace(0, 5, 2); axes[0].plot(t, w * t + b, color="tab:orange")
axes[1].plot(verluste); axes[1].set_yscale("log"); axes[1].set_xlabel("Epoche"); axes[1].set_ylabel("MSE")
plt.show()

## 5. Batch vs. Mini-Batch vs. SGD

**Aufgabe 5.1:** Implementiere Mini-Batch-GD: In jeder Epoche die Daten mischen und in Gruppen der Größe `batch` durchgehen.
Mit `batch = n` ist das Batch-GD, mit `batch = 1` SGD. Vergleiche die Lernkurven über die **Anzahl Updates**.

In [ ]:
def train_minibatch(x, y, lr, epochen, batch, seed=0):
    r_gen = np.random.default_rng(seed)
    w, b = 0.0, 0.0
    verluste = []
    for _ in range(epochen):
        idx = r_gen.permutation(len(x))
        for start in range(0, len(x), batch):
            # TODO: Mini-Batch auswählen, Gradient auf dem Batch berechnen, Update; danach Gesamt-MSE anhängen
            ...
    return w, b, verluste

for batch, lr in [(n, 0.05), (32, 0.05), (1, 0.01)]:
    w_, b_, v = train_minibatch(x, y, lr, epochen=20, batch=batch)
    plt.plot(v[:600], label=f"Batchgröße {batch}: w={w_:.2f}, b={b_:.2f}")
plt.yscale("log"); plt.xlabel("Anzahl Updates"); plt.ylabel("MSE (alle Daten)"); plt.legend(); plt.show()

**Beobachtung:** Kleine Batches machen viel mehr Updates pro Epoche und kommen deshalb schneller voran – dafür ist die Kurve verrauscht.

## 6. Vergleich mit scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression, SGDRegressor

lin = LinearRegression().fit(x.reshape(-1, 1), y)
sgd = SGDRegressor(max_iter=1000, tol=1e-6, random_state=0).fit(x.reshape(-1, 1), y)
print("LinearRegression:", lin.coef_[0], lin.intercept_)
print("SGDRegressor:    ", sgd.coef_[0], sgd.intercept_[0])
print("unser GD:        ", w, b)